# Hukom AI Phase 2: Training Pipeline

## Overview
This notebook implements the training pipeline for the Hukom-AI legal text classification model. It provides two training strategies:
1. **Strategy A (Head+Tail)**: Fast training using the first 128 and last 382 tokens.
2. **Strategy B (Sliding Window)**: Thorough training using a sliding window approach over the entire text.

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas

import os
# Create directory for data if it doesn't exist
if not os.path.exists("hukom_corpus_friendly"):
    os.makedirs("hukom_corpus_friendly")
print("Environment Setup Complete.")

In [ ]:
%%writefile hukom_utils.py
import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from transformers import Trainer

# --- CONFIGURATION ---
DATA_CSV = "hukom_dataset_final.csv"
CORPUS_DIR = "hukom_corpus_friendly"

def load_and_split_data():
    """
    Loads CSV, filters out -1 (Unknowns), and splits into Train/Val/Test.
    """
    if not os.path.exists(DATA_CSV):
        print(f"❌ Error: {DATA_CSV} not found. Please upload it.")
        return None, None, None

    print(f"Loading {DATA_CSV}...")
    df = pd.read_csv(DATA_CSV)
    
    # 1. Filter out Unknowns (-1)
    df = df[df['label'] != -1]
    
    # 2. Map filename to full path
    df['text_path'] = df['filename'].apply(lambda x: os.path.join(CORPUS_DIR, x))
    
    # 3. Read text content
    texts = []
    labels = []
    print("Reading text files...")
    for idx, row in df.iterrows():
        try:
            with open(row['text_path'], "r", encoding="utf-8") as f:
                content = f.read()
                if "========== FACTS ==========" in content:
                    parts = content.split("========== FACTS ==========")
                    if "========== RULING ==========" in parts[1]:
                        facts = parts[1].split("========== RULING ==========")[0].strip()
                        texts.append(facts)
                        labels.append(row['label'])
        except Exception:
            continue

    # 4. Split (80% Train, 10% Val, 10% Test)
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )
    val_texts, test_texts, val_labels, test_labels = train_test_split(
        test_texts, test_labels, test_size=0.5, stratify=test_labels, random_state=42
    )
    
    print(f"Data Loaded: {len(train_texts)} Train, {len(val_texts)} Val, {len(test_texts)} Test")
    return (train_texts, train_labels), (val_texts, val_labels), (test_texts, test_labels)

class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
            
        return (loss, outputs) if return_outputs else loss

In [ ]:
%%writefile train_headtail.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from hukom_utils import load_and_split_data, CustomTrainer

# --- CONFIG ---
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 3

class HeadTailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        tokens = self.tokenizer(text, truncation=False, return_tensors='pt')
        input_ids = tokens['input_ids'][0]
        
        if len(input_ids) > self.max_len:
            head = input_ids[:129] 
            tail = input_ids[-(self.max_len - 129):] 
            input_ids = torch.cat([head, tail])
        else:
            padding = torch.zeros(self.max_len - len(input_ids), dtype=torch.long)
            input_ids = torch.cat([input_ids, padding])

        attention_mask = (input_ids != 0).long()
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.long)
        }

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1}

if __name__ == "__main__":
    (train_txt, train_lbl), (val_txt, val_lbl), _ = load_and_split_data()
    if train_txt is None: exit()

    class_weights = compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    print(f"Class Weights: {class_weights}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

    train_dataset = HeadTailDataset(train_txt, train_lbl, tokenizer)
    val_dataset = HeadTailDataset(val_txt, val_lbl, tokenizer)

    training_args = TrainingArguments(
        output_dir='./results_headtail',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        fp16=True,
        load_best_model_at_end=True,
    )

    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )

    trainer.train()
    trainer.save_model("./hukom_model_headtail")
    print("Strategy A Complete!")

In [ ]:
%%writefile train_sliding.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from hukom_utils import load_and_split_data, CustomTrainer

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 2 

def tokenize_sliding_window(texts, labels, tokenizer):
    all_input_ids = []
    all_masks = []
    all_labels = []
    
    encodings = tokenizer(
        texts, 
        truncation=True, 
        padding="max_length", 
        max_length=MAX_LEN, 
        return_overflowing_tokens=True, 
        stride=128, 
        return_tensors="pt"
    )
    sample_map = encodings.pop("overflow_to_sample_mapping")
    
    for i, sample_idx in enumerate(sample_map):
        all_input_ids.append(encodings['input_ids'][i])
        all_masks.append(encodings['attention_mask'][i])
        all_labels.append(labels[sample_idx])

    return all_input_ids, all_masks, all_labels

class SlidingDataset(Dataset):
    def __init__(self, input_ids, masks, labels):
        self.input_ids = input_ids
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.masks[idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1}

if __name__ == "__main__":
    (train_txt, train_lbl), (val_txt, val_lbl), _ = load_and_split_data()
    if train_txt is None: exit()

    class_weights = compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ids, train_masks, train_labels_exp = tokenize_sliding_window(train_txt, train_lbl, tokenizer)
    val_ids, val_masks, val_labels_exp = tokenize_sliding_window(val_txt, val_lbl, tokenizer)
    
    train_dataset = SlidingDataset(train_ids, train_masks, train_labels_exp)
    val_dataset = SlidingDataset(val_ids, val_masks, val_labels_exp)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

    training_args = TrainingArguments(
        output_dir='./results_sliding',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        fp16=True, 
        load_best_model_at_end=True,
    )

    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )

    trainer.train()
    trainer.save_model("./hukom_model_sliding")
    print("Strategy B Complete!")

## Instructions

### Prerequisites
1. **Dataset**: Ensure you have `hukom_dataset_final.csv`.
2. **Corpus**: Ensure you have the `hukom_corpus_friendly` folder (zipped or unzipped).

### Execution Steps
1. Upload your data files to the Colab environment.
2. Run the **Setup & Installation** cell to install dependencies.
3. Run the **Script Generation** cells to create `hukom_utils.py`, `train_headtail.py`, and `train_sliding.py`.
4. Choose and run a **Training Strategy** in the final cell.

In [ ]:
# Run Strategy A (Fast)
!python train_headtail.py

# Run Strategy B (Thorough - Uncomment to run)
# !python train_sliding.py